# 04 — MCMC Fitting a Prism Spectrum

This notebook demonstrates the `jwspecmcmc` MCMC workflow:
1. Load a JWST NIRSpec PRISM spectrum
2. Fit emission lines with `emcee` (MCMC) instead of least-squares
3. Compare MCMC posteriors with bootstrap uncertainties from `jwspecfit`
4. Inspect asymmetric credible intervals
5. Compute flux-ratio posteriors for line-ratio diagnostics
6. Diagnostic plots: corner, traces, flux posteriors
7. Use `to_fit_result()` for compatibility with `jwspecfit.plot_fit()`

The MCMC sampler explores the same parameter space as the least-squares
fitter, but gives proper posterior distributions and asymmetric uncertainties.

In [ ]:
import jwspecfit
import jwspecmcmc
import matplotlib.pyplot as plt
import numpy as np

print(f"jwspecfit  v{jwspecfit.__version__}")
print(f"jwspecmcmc v{jwspecmcmc.__version__}")

## Load the spectrum

Same PRISM spectrum as notebook 01.

In [ ]:
spec = jwspecfit.read_fits("../../data/borg-v4_prism-clear_1747_732.spec.fits", z=8.22288)

print(f"Grating:    {spec.grating}")
print(f"Pixels:     {spec.n_pix}")
print(f"Wave range: {spec.wave_um.min():.3f} \u2013 {spec.wave_um.max():.3f} \u00b5m")

## MCMC fit with emcee

A single call mirrors `jwspecfit.fit_lines()` but uses MCMC sampling.
By default it runs a quick MLE fit first to initialise the walkers,
then samples with emcee.

For a quick test, use fewer steps. For publication, use
`n_steps=5000` or more and check convergence.

In [ ]:
result = jwspecmcmc.fit_lines(
    spec, z=8.22288,
    sampler="emcee",
    n_steps=2000,
    n_walkers=64,
    seed=42,
)

print(f"Sampler:     {result.sampler_name}")
print(f"Lines:       {', '.join(result.lines.keys())}")
print(f"Chain shape: {result.flat_chains.shape}")
print(f"Burn-in:     {result.sampler_meta.get('n_burn', '?')} steps")

## Per-line results with asymmetric uncertainties

MCMC gives proper (16th, 84th) percentile credible intervals,
which can be asymmetric for parameters near bounds.

In [ ]:
print(f"{'Line':<18s} {'Flux':>12s} {'\u2212err':>12s} {'+err':>12s} {'SNR':>8s} {'EW (\u00c5)':>10s}")
print("-" * 76)
for name, lr in result.lines.items():
    print(
        f"{name:<18s} {lr.flux:12.3e} {lr.flux_err[0]:12.3e} {lr.flux_err[1]:12.3e}"
        f" {lr.snr:8.1f} {lr.ew_A:10.1f}"
    )

## Convergence diagnostics

The Gelman\u2013Rubin R-hat should be < 1.05 and the effective sample
size (ESS) should be > 100 for all parameters.

In [ ]:
conv = result.convergence
print(f"R-hat max:  {conv.get('r_hat_max', 'N/A'):.3f}")
print(f"ESS min:    {conv.get('ess_min', 'N/A'):.0f}")
print(f"Converged:  {conv.get('converged', 'N/A')}")

## Trace plots

Inspect how the chains explore parameter space.  Good chains
should look like "hairy caterpillars" with no trends.

In [ ]:
fig = jwspecmcmc.plot_traces(
    result,
    params=["A_OIII_5007", "A_HBETA", "sigma_OIII_5007"],
)
plt.show()

## Flux posterior histograms

The full posterior distribution of the [OIII] 5007 flux, with
median and 68% credible interval.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, line in zip(axes, ["OIII_5007", "OIII_4959", "HBETA"]):
    if line in result.lines:
        jwspecmcmc.plot_flux_posterior(result, line, ax=ax)

plt.tight_layout()
plt.show()

## Corner plot

Joint posterior of selected parameters.  Off-diagonal panels show
covariances between parameters.

In [ ]:
fig = jwspecmcmc.plot_corner(
    result,
    params=["A_OIII_5007", "A_OIII_4959", "A_HBETA", "sigma_OIII_5007"],
)
plt.show()

## Flux-ratio posteriors

Compute the posterior on O3/H\u03b2 = [OIII] 5007 / H\u03b2,
which is a key diagnostic for metallicity and ionisation.

In [ ]:
if "OIII_5007" in result.lines and "HBETA" in result.lines:
    ratio = result.flux_ratio_posterior("OIII_5007", "HBETA")
    ratio_clean = ratio[np.isfinite(ratio)]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(ratio_clean, bins=50, density=True, alpha=0.7, color="C2", edgecolor="C2")
    med = np.median(ratio_clean)
    lo, hi = np.percentile(ratio_clean, [16, 84])
    ax.axvline(med, color="C1", ls="-", lw=1.5, label=f"Median = {med:.2f}")
    ax.axvline(lo, color="C1", ls="--", lw=1.0, label=f"16th = {lo:.2f}")
    ax.axvline(hi, color="C1", ls="--", lw=1.0, label=f"84th = {hi:.2f}")
    ax.set_xlabel(r"[OIII] 5007 / H$\beta$")
    ax.set_ylabel("Probability density")
    ax.legend(fontsize=9)
    ax.set_title(r"O3/H$\beta$ flux ratio posterior")
    plt.tight_layout()
    plt.show()
else:
    print("OIII_5007 or HBETA not in fitted lines.")

## Compare with least-squares fit

Run the standard `jwspecfit.fit_lines()` for comparison.  The MCMC
result can be converted to a `FitResult` for `plot_fit()` compatibility.

In [ ]:
result_lsq = jwspecfit.fit_lines(spec, z=8.22288, n_boot=0)

print(f"{'Line':<18s} {'Flux (LSQ)':>12s} {'Flux (MCMC)':>12s} {'\u0394(%)':>8s}")
print("-" * 54)
for name in result_lsq.lines:
    if name in result.lines:
        f_lsq = result_lsq.lines[name].flux
        f_mcmc = result.lines[name].flux
        delta = 100 * (f_mcmc - f_lsq) / f_lsq if f_lsq != 0 else 0
        print(f"{name:<18s} {f_lsq:12.3e} {f_mcmc:12.3e} {delta:8.1f}")

## Plot the MCMC fit using jwspecfit.plot_fit()

`to_fit_result()` converts the MCMC median posterior to a standard
`FitResult` that works with all `jwspecfit` plotting functions.

In [ ]:
fit_result = result.to_fit_result()

print(f"\u03c7\u00b2/dof = {fit_result.chi2:.2f}")

fig = jwspecfit.plot_fit(fit_result)
fig.suptitle("MCMC median posterior fit", y=1.02)
plt.show()

## Fit specific lines only

Restrict to [OIII] + H\u03b2 for faster MCMC exploration.

In [ ]:
result_o3 = jwspecmcmc.fit_lines(
    spec, z=8.22288,
    lines=["OIII_4959", "OIII_5007", "HBETA"],
    n_steps=1000,
    progress=True,
)

for name, lr in result_o3.lines.items():
    print(f"  {name:<18s} flux={lr.flux:.3e} (-{lr.flux_err[0]:.3e}, +{lr.flux_err[1]:.3e})")

## Custom priors

Override the default uniform priors with Gaussian priors on specific
parameters.  Useful when you have prior knowledge from other observations.

In [ ]:
from jwspecmcmc import GaussianPrior

# Example: informative Gaussian prior on OIII_5007 amplitude
result_prior = jwspecmcmc.fit_lines(
    spec, z=8.22288,
    lines=["OIII_4959", "OIII_5007", "HBETA"],
    n_steps=1000,
    prior_overrides={
        "A_OIII_5007": GaussianPrior(mean=8e-18, std=2e-18, lo=0, hi=1e-15),
    },
)

print("With Gaussian prior on A_OIII_5007:")
for name, lr in result_prior.lines.items():
    print(f"  {name:<18s} flux={lr.flux:.3e} (-{lr.flux_err[0]:.3e}, +{lr.flux_err[1]:.3e})")